In [7]:
import pandas as pd

base = "https://huggingface.co/datasets/Mouwiya/cfpb-consumer-complaints/resolve/main/data/complaints-{:05d}.parquet"
cols = ["date_received", "product", "sub_product", "issue", "sub_issue",
        "consumer_complaint_narrative", "company", "state", "tags",
        "company_response_to_consumer", "timely_response", "complaint_id"]

keep = []
i = 0
while True:
    try:
        part = pd.read_parquet(base.format(i), columns=cols)
    except Exception as e:
        print(f"\nStopped at file {i}: {type(e).__name__}")
        break
    part = part[(part["product"] == "Credit card") & part["consumer_complaint_narrative"].notna()]
    keep.append(part)
    print(f"{i}:{len(part)}", end="  ")
    i += 1

df = pd.concat(keep, ignore_index=True)
df["date_received"] = pd.to_datetime(df["date_received"], errors="coerce")
print("\nAll credit card complaints with narrative:", df.shape)
print("Date range:", df["date_received"].min(), "to", df["date_received"].max())

df = df[df["date_received"].dt.year == 2025]
print("2025 only:", df.shape)
df.to_csv("data/raw/complaints_raw.csv", index=False)

0:1016  1:374  2:440  3:1201  4:513  5:361  6:0  7:1087  8:7148  9:5648  10:4743  11:4876  12:4770  13:3267  14:2816  15:2662  16:2516  17:2331  18:2351  19:2262  20:2470  21:2030  22:2101  23:2242  24:1855  25:1815  26:1852  27:1952  28:2111  29:1975  30:2115  31:1954  32:1977  33:1844  34:2010  35:2009  36:1768  37:1547  38:1900  39:1894  40:1605  41:1039  42:2168  43:1445  44:1744  45:949  46:1591  47:1229  48:1104  49:1036  50:369  51:157  52:518  53:0  54:304  55:18015  56:0  57:0  58:0  59:0  60:0  61:0  62:0  63:0  64:0  65:0  66:0  
Stopped at file 67: HTTPError

All credit card complaints with narrative: (123076, 12)
Date range: 2023-09-20 19:18:04+00:00 to 2026-06-17 11:11:22+00:00
2025 only: (51, 12)


In [8]:
df = pd.concat(keep, ignore_index=True)

print("Sample raw dates:", df["date_received"].astype(str).sample(8, random_state=1).tolist())

df["date_received"] = pd.to_datetime(df["date_received"], errors="coerce", format="mixed", utc=True)
print("Dates that still couldn't be read:", df["date_received"].isna().sum())
print(df["date_received"].dt.year.value_counts().sort_index())

Sample raw dates: ['2015-05-09 00:00:00', '2024-03-04 00:00:00', '2024-01-05 00:00:00', '2015-08-30 00:00:00', '2024-02-28 00:00:00', '2024-10-22 00:00:00', '2015-09-14 00:00:00', '2024-02-07 00:00:00']
Dates that still couldn't be read: 0
date_received
2015     6243
2016     9433
2017     3161
2023    12905
2024    35451
2025    41526
2026    14357
Name: count, dtype: int64


In [9]:
df = df[df["date_received"].dt.year >= 2023].copy()
print("Duplicate complaint IDs:", df["complaint_id"].duplicated().sum())
df = df.drop_duplicates(subset="complaint_id")

df = df.rename(columns={"consumer_complaint_narrative": "narrative"})
print("Final:", df.shape)
df.to_csv("data/raw/complaints_raw.csv", index=False)

Duplicate complaint IDs: 0
Final: (104239, 12)


In [10]:
df["issue"].value_counts()

issue
Problem with a purchase shown on your statement                    27066
Other features, terms, or problems                                 11978
Getting a credit card                                              11814
Fees or interest                                                   11435
Problem with a company's investigation into an existing problem     8993
Closing your account                                                7801
Incorrect information on your report                                6956
Problem when making payments                                        6028
Advertising and marketing, including promotional offers             5072
Trouble using your card                                             4385
Struggling to pay your bill                                         1307
Improper use of your report                                          963
Credit monitoring or identity theft protection services              275
Problem with fraud alerts or security freezes

In [11]:
df["words"] = df["narrative"].str.split().str.len()
print(df["words"].describe().round(0))
print("\n--- Example complaint ---\n")
print(df["narrative"].iloc[0][:1000])

count    104239.0
mean        210.0
std         228.0
min           2.0
25%          71.0
50%         157.0
75%         270.0
max        6469.0
Name: words, dtype: float64

--- Example complaint ---

Credit card issuer : Imprint ( XXXX XXXX XXXX and XXXX XXXX XXXX, formerly XXXX XXXX  ) I am filing this complaint due to an improper interest charge and lack of resolution following the migration of my XXXX XXXX XXXX credit card account from XXXX XXXX  to Imprint. 

I originally had a XXXXXXXX XXXX XXXXXXXX XXXX credit card that I only used a few times. My most recent XXXX statement was dated XX/XX/year>, showing a {$0.00} balance. I had paid my previous XXXX statement in full and on time. Therefore, I had no outstanding balance or trailing interest at the time of migration. 

XXXXXXXX XXXX XXXXXXXX notified customers that the credit card program would migrate from XXXX to Imprint on XX/XX/year>, and that balances, purchases, and credits would transfer automatically. They said customers d